<a href="https://colab.research.google.com/github/clee2026/MSDS_498_Capstone/blob/main/nlp_categories/02_nyc311_apply_issue_categories_full_parquet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Upload these files:

* nlp_issue_kmeans_model.joblib
* nlp_issue_category_config.json
* nlp_issue_cluster_label_mapping.csv

In [ ]:
# 0. Install packages
!pip install -q pyarrow pandas scikit-learn sentence-transformers joblib tqdm

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration

import os, json
from pathlib import Path

PARQUET_PATH = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet"

NLP_OUTPUT_DIR = "/content/nlp_issue_outputs"
# NLP_OUTPUT_DIR = "/content/drive/MyDrive/project_data/Capstone Course Project Files/outputs_nlp_issue_categories"

MODEL_PATH = "/content/nlp_issue_kmeans_model.joblib"
CONFIG_PATH = "/content/nlp_issue_category_config.json"
LABEL_MAPPING_PATH = "/content/nlp_issue_cluster_label_mapping.csv"

# MODEL_PATH = os.path.join(NLP_OUTPUT_DIR, "nlp_issue_kmeans_model.joblib")
# CONFIG_PATH = os.path.join(NLP_OUTPUT_DIR, "nlp_issue_category_config.json")
# LABEL_MAPPING_PATH = os.path.join(NLP_OUTPUT_DIR, "nlp_issue_cluster_label_mapping.csv")

FULL_OUTPUT_PATH = "/content/nlp_issue_outputs/nyc311_with_nlp_issue_categories.parquet"
SUMMARY_OUTPUT_PATH = "/content/nlp_issue_outputs/full_issue_category_counts.csv"

# FULL_OUTPUT_PATH = os.path.join(NLP_OUTPUT_DIR, "nyc311_with_nlp_issue_categories.parquet")
# SUMMARY_OUTPUT_PATH = os.path.join(NLP_OUTPUT_DIR, "full_issue_category_counts.csv")

PARQUET_BATCH_SIZE = 50_000

# For testing, set APPLY_LIMIT to something like 200_000.
# For the full run, set APPLY_LIMIT = None.
APPLY_LIMIT = 200_000

In [ ]:
from pathlib import Path
Path(NLP_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Load model and config

import joblib
import pandas as pd
import pyarrow.parquet as pq

kmeans = joblib.load(MODEL_PATH)

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

available_text_columns = config["available_text_columns"]
EMBEDDING_MODEL_NAME = config["embedding_model_name"]

print("Using text columns:", available_text_columns)
print("Embedding model:", EMBEDDING_MODEL_NAME)

In [ ]:
# 4. Load optional manual label mapping

if os.path.exists(LABEL_MAPPING_PATH):
    label_map_df = pd.read_csv(LABEL_MAPPING_PATH)
    cluster_to_label = dict(zip(label_map_df["issue_cluster"], label_map_df["final_issue_category"]))
    print("Loaded manual label mapping:", LABEL_MAPPING_PATH)
else:
    cluster_to_label = {}
    print("No manual label mapping found. Output will include numeric clusters only.")

In [ ]:
# 5. Text preparation functions

import re

DOMAIN_STOPWORDS = {
    "please", "call", "called", "complaint", "customer", "service",
    "request", "reported", "information", "provided", "will", "may",
    "nyc", "city", "department"
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if len(t) > 2 and t not in DOMAIN_STOPWORDS]
    return " ".join(tokens)

def build_combined_text(df_batch):
    parts = []
    for col in ["complaint_type", "descriptor", "resolution_description", "agency_name", "location_type", "borough"]:
        if col in df_batch.columns:
            parts.append(df_batch[col].fillna("").astype(str))

    combined = parts[0]
    for part in parts[1:]:
        combined = combined + " " + part

    return combined.str.strip()

In [ ]:
# 6. Apply categories in parquet batches

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

pf = pq.ParquetFile(PARQUET_PATH)

columns_to_read = [c for c in available_text_columns if c in pf.schema.names]

# Keep an ID if available.
possible_id_columns = ["unique_key", "created_date", "closed_date", "complaint_type", "descriptor", "borough"]
passthrough_columns = [c for c in possible_id_columns if c in pf.schema.names and c not in columns_to_read]

read_columns = list(dict.fromkeys(columns_to_read + passthrough_columns))

writer = None
processed_rows = 0
category_counts = {}

for batch in tqdm(pf.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=read_columns)):
    df_batch = batch.to_pandas()

    if APPLY_LIMIT is not None:
        remaining = APPLY_LIMIT - processed_rows
        if remaining <= 0:
            break
        df_batch = df_batch.head(remaining)

    df_batch["combined_text"] = build_combined_text(df_batch)
    df_batch["clean_text"] = df_batch["combined_text"].apply(clean_text)

    embeddings = model.encode(
        df_batch["clean_text"].tolist(),
        batch_size=256,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    df_batch["issue_cluster"] = kmeans.predict(embeddings)

    if cluster_to_label:
        df_batch["issue_category"] = df_batch["issue_cluster"].map(cluster_to_label).fillna("Unmapped cluster")
    else:
        df_batch["issue_category"] = "Cluster " + df_batch["issue_cluster"].astype(str)

    counts = df_batch["issue_category"].value_counts().to_dict()
    for category, count in counts.items():
        category_counts[category] = category_counts.get(category, 0) + int(count)

    output_cols = passthrough_columns + ["issue_cluster", "issue_category", "clean_text"]
    output_cols = [c for c in output_cols if c in df_batch.columns]

    table = pa.Table.from_pandas(df_batch[output_cols], preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(FULL_OUTPUT_PATH, table.schema)

    writer.write_table(table)

    processed_rows += len(df_batch)

    if APPLY_LIMIT is not None and processed_rows >= APPLY_LIMIT:
        break

if writer is not None:
    writer.close()

print("Processed rows:", processed_rows)
print("Saved:", FULL_OUTPUT_PATH)

In [ ]:
# 7. Save full category count summary

summary_counts_df = (
    pd.DataFrame(
        [{"issue_category": k, "record_count": v} for k, v in category_counts.items()]
    )
    .sort_values("record_count", ascending=False)
)

summary_counts_df["share"] = summary_counts_df["record_count"] / summary_counts_df["record_count"].sum()

summary_counts_df.to_csv(SUMMARY_OUTPUT_PATH, index=False)

display(summary_counts_df)
print("Saved:", SUMMARY_OUTPUT_PATH)

In [ ]:
# ZIP OUTPUTS


import shutil

ZIP_PATH = "/content/nlp_issue_full_results.zip"

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=NLP_OUTPUT_DIR
)

print("ZIP created:", ZIP_PATH)